# EVOLVE – Model Training

This notebook trains and adapts a YOLOv8 object detection model on the EVOLVE training dataset.

The objective is to fine-tune a pre-trained detector under extreme low-light and volatile visual conditions.

## Experimental Setup

- Architecture: YOLOv8 (Ultralytics)
- Initialization: COCO pre-trained weights
- Dataset: EVOLVE (manually annotated, stratified by luminance)
- Training split: defined in `configs/dataset.yaml`
- Validation split: used for model selection

This notebook performs:

1. Baseline evaluation of a pre-trained YOLOv8 model  
2. Fine-tuning on the EVOLVE training dataset  
3. Validation-based model selection  

The test set is **not used** in this notebook.

In [ ]:
import torch
from ultralytics import YOLO
import yaml
import os

# Reproducibility
SEED = 42
torch.manual_seed(SEED)

# Device selection
device = 0 if torch.cuda.is_available() else "cpu"
print("Device used:", device)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Dataset Configuration

The dataset split is defined in `configs/dataset.yaml`.

We verify:

- Train path
- Validation path
- Number of classes
- Class names

We verify the configuration before training.

In [ ]:
DATASET_CONFIG_PATH = "../configs/dataset.yaml"

with open(DATASET_CONFIG_PATH, "r") as f:
    dataset_config = yaml.safe_load(f)

print("Dataset configuration loaded.\n")
print("Train path:", dataset_config.get("train"))
print("Val path:", dataset_config.get("val"))
print("Number of classes:", dataset_config.get("nc"))
print("Class names:", dataset_config.get("names"))

## Baseline Evaluation (Pre-trained Model)

We evaluate the pre-trained YOLOv8 model (COCO weights) without any fine-tuning on the EVOLVE dataset.

This establishes a reference performance level before adaptation.

In [ ]:
baseline_model = YOLO("yolov8n.pt")

baseline_metrics = baseline_model.val(
    data=DATASET_CONFIG_PATH,
    split="val",
    device=device
)

print("\nBaseline Validation Results:")
print("mAP50-95:", baseline_metrics.box.map)
print("mAP50:", baseline_metrics.box.map50)
print("Precision:", baseline_metrics.box.mp)
print("Recall:", baseline_metrics.box.mr)

## Fine-Tuning on EVOLVE

We fine-tune the YOLOv8 model on the EVOLVE training dataset.

Training parameters:

- Epochs: 50
- Image size: 640
- Batch size: 16
- Initialization: COCO pre-trained weights

In [ ]:
model = YOLO("yolov8n.pt")

results = model.train(
    data=DATASET_CONFIG_PATH,
    epochs=50,
    imgsz=640,
    batch=16,
    device=device,
    seed=SEED,
    project="../runs/train",
    name="evolve_yolov8n",
    exist_ok=True
)

## Validation Performance After Fine-Tuning

We now evaluate the fine-tuned model on the validation set.

This allows direct comparison with the baseline.

In [ ]:
finetuned_model = YOLO(results.save_dir / "weights" / "best.pt")

finetuned_metrics = finetuned_model.val(
    data=DATASET_CONFIG_PATH,
    split="val",
    device=device
)

print("\nFine-Tuned Validation Results:")
print("mAP50-95:", finetuned_metrics.box.map)
print("mAP50:", finetuned_metrics.box.map50)
print("Precision:", finetuned_metrics.box.mp)
print("Recall:", finetuned_metrics.box.mr)

## Baseline vs Fine-Tuned Comparison

We compare validation metrics to assess the impact of domain adaptation.

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    "Metric": ["mAP50-95", "mAP50", "Precision", "Recall"],
    "Baseline": [
        baseline_metrics.box.map,
        baseline_metrics.box.map50,
        baseline_metrics.box.mp,
        baseline_metrics.box.mr
    ],
    "Fine-Tuned": [
        finetuned_metrics.box.map,
        finetuned_metrics.box.map50,
        finetuned_metrics.box.mp,
        finetuned_metrics.box.mr
    ]
})

comparison

## Model Selection

The checkpoint corresponding to the best validation performance is automatically retrieved from the training directory and exported to:

`../results/best_model.pt`

This model will be evaluated on the test set in the evaluation notebook.

The test set remains untouched during training.

In [ ]:
# Export baseline + fine-tuned metrics

import json
import os

os.makedirs("../results", exist_ok=True)

baseline_dict = {
    "mAP50-95": float(baseline_metrics.box.map),
    "mAP50": float(baseline_metrics.box.map50),
    "Precision": float(baseline_metrics.box.mp),
    "Recall": float(baseline_metrics.box.mr)
}

finetuned_dict = {
    "mAP50-95": float(finetuned_metrics.box.map),
    "mAP50": float(finetuned_metrics.box.map50),
    "Precision": float(finetuned_metrics.box.mp),
    "Recall": float(finetuned_metrics.box.mr)
}

with open("../results/baseline_metrics.json", "w") as f:
    json.dump(baseline_dict, f, indent=4)

with open("../results/finetuned_metrics.json", "w") as f:
    json.dump(finetuned_dict, f, indent=4)

print("Metrics exported successfully.")

In [ ]:
# Export comparison CSV

comparison.to_csv("../results/training_summary.csv", index=False)

In [ ]:
# Export best model dynamically

import shutil

best_model_path = results.save_dir / "weights" / "best.pt"

shutil.copy(best_model_path, "../results/best_model.pt")

print("Best model exported from:", best_model_path)

In [ ]:
# Export config training

from pathlib import Path

train_images = list(Path(dataset_config["train"]).glob("*.jpg"))
val_images = list(Path(dataset_config["val"]).glob("*.jpg"))

training_metadata = {
    "model_architecture": "yolov8n"
    "epochs": 50,
    "imgsz": 640,
    "batch_size": 16,
    "seed": SEED,
    "train_images": len(train_images),
    "val_images": len(val_images)
}

with open("../results/training_config.json", "w") as f:
    json.dump(training_metadata, f, indent=4)

## Conclusion

This notebook demonstrated:

- Baseline performance of a pre-trained detector
- Performance after domain-specific fine-tuning
- Controlled comparison on validation data

The next step consists of evaluating the selected model on the held-out test set.